# block-group-stack — worked example 3: Chain several BlockGroups into a mini CNN trunk

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `block-group-stack`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A full ResNet body is a sequence of BlockGroups, each itself a BlockGroup-stack. Group `k` takes the previous group's `out_feats` as its `in_feats`, doubles the channels, and downsamples once with `first_stride=2`. Chaining groups multiplies the per-group invariant: each group changes shape exactly once at its own block 0.

## Worked solution

We compose the stacking pattern one level up.

**Step 1 - one group builder.** Reuse the canonical rule: block 0 is `(in, out, stride)`, the rest `(out, out, 1)`.

**Step 2 - chain groups.** Starting from `in_feats`, we walk a list of `out_feats` per group. For each group we feed the running `cur_in` as `in_feats`, set its `out_feats` from the list, and use `first_stride=2` so each group halves the spatial size. After building, `cur_in` advances to that group's `out_feats` so the next group's input channels match - this hand-off is the key to a legal chain.

**Step 3 - wrap everything in an outer `nn.Sequential`.** A `nn.Sequential` of `nn.Sequential`s is still a valid module; calling it runs the groups in order.

**Step 4 - verify.** With 3 groups of `first_stride=2` each, a `32x32` input becomes `4x4` (halved three times) and channels follow the widths list. We print the shape and assert it. This shows the BlockGroup-stack pattern composes cleanly into a CNN trunk.

In [ ]:
import torch as t
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)

def make_block_group(in_feats, out_feats, n_blocks, first_stride):
    blocks = [ResBlock(in_feats, out_feats, first_stride=first_stride)]
    for _ in range(n_blocks - 1):
        blocks.append(ResBlock(out_feats, out_feats, first_stride=1))
    return nn.Sequential(*blocks)

def make_trunk(in_feats, group_widths, blocks_per_group):
    groups = []
    cur_in = in_feats
    for w in group_widths:
        groups.append(make_block_group(cur_in, w, blocks_per_group, first_stride=2))
        cur_in = w
    return nn.Sequential(*groups)

t.manual_seed(0)
trunk = make_trunk(in_feats=4, group_widths=[16, 32, 64], blocks_per_group=2)
x = t.randn(1, 4, 32, 32)
y = trunk(x)
print('num groups :', len(trunk))
print('output shape:', tuple(y.shape))
assert y.shape == (1, 64, 4, 4)
print('three groups, three halvings, channels 4 -> 64: OK')